In [1]:
import pandas as pd
import numpy as np
import re
import difflib
import importlib
import cleaning as cln
importlib.reload(cln)


<module 'cleaning' from 'd:\\DS108\\DS108_Phone-Reccomendation\\cleaning.py'>

### IMPORTING

In [2]:
cps = pd.read_csv(r'cellphones_raw.csv')
cps.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 50 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Unnamed                 966 non-null    int64
 1    0                      966 non-null    int64
 2   Name                    966 non-null    str  
 3   Price                   333 non-null    str  
 4   Link                    966 non-null    str  
 5   Kích thước màn hình     873 non-null    str  
 6   Công nghệ màn hình      814 non-null    str  
 7   Camera sau              855 non-null    str  
 8   Camera trước            823 non-null    str  
 9   Chipset                 857 non-null    str  
 10  Công nghệ NFC           762 non-null    str  
 11  Bộ nhớ trong            915 non-null    str  
 12  Thẻ SIM                 696 non-null    str  
 13  Hệ điều hành            762 non-null    str  
 14  Độ phân giải màn hình   659 non-null    str  
 15  Tính năng màn hình      732 non-nu

In [3]:
cps = cps.drop(columns=['Unnamed', ' 0'])

In [4]:
df = cps.copy()

DROP NHỮNG THUỘC TÍNH CÓ HƠN 50% LÀ NULL (TRỪ PRICE VÀ TẦN SỐ QUÉT)

In [5]:
thresh_limit = 0.5 * df.shape[0]

cols_to_check = [col for col in df.columns if col != "Tần số quét"]

cols_to_keep = (
    df[cols_to_check].dropna(thresh=thresh_limit, axis=1).columns.tolist()
)

cols_to_keep.append("Tần số quét")

df = df[cols_to_keep]

In [6]:
df.columns

Index(['Name', 'Link', 'Kích thước màn hình', 'Công nghệ màn hình',
       'Camera sau', 'Camera trước', 'Chipset', 'Công nghệ NFC',
       'Bộ nhớ trong', 'Thẻ SIM', 'Hệ điều hành', 'Độ phân giải màn hình',
       'Tính năng màn hình', 'Loại CPU', 'Dung lượng RAM', 'Pin',
       'Tần số quét'],
      dtype='str')

 ĐỐI TÊN THUỘC TÍNH 

In [7]:
feature_mapping = {
    "Tên": "Name",
    "Link": "Link",
    "Kích thước màn hình": "Screen Size",
    "Công nghệ màn hình": "Display",
    "Camera sau": "Rear Camera",
    "Camera trước": "Front Camera",
    "Chipset": "Chipset",
    "Công nghệ NFC": "NFC",
    "Bộ nhớ trong": "ROM",
    "Thẻ SIM": "SIM Card",
    "Hệ điều hành": "Operating System",
    "Độ phân giải màn hình": "Screen Resolution",
    "Tính năng màn hình": "Display Features",
    "Loại CPU": "CPU",
    "Dung lượng RAM": "RAM",
    "Pin": "Battery",
}

df = df.rename(columns=feature_mapping)


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               966 non-null    str  
 1   Link               966 non-null    str  
 2   Screen Size        873 non-null    str  
 3   Display            814 non-null    str  
 4   Rear Camera        855 non-null    str  
 5   Front Camera       823 non-null    str  
 6   Chipset            857 non-null    str  
 7   NFC                762 non-null    str  
 8   ROM                915 non-null    str  
 9   SIM Card           696 non-null    str  
 10  Operating System   762 non-null    str  
 11  Screen Resolution  659 non-null    str  
 12  Display Features   732 non-null    str  
 13  CPU                588 non-null    str  
 14  RAM                870 non-null    str  
 15  Battery            869 non-null    str  
 16  Tần số quét        206 non-null    str  
dtypes: str(17)
memory usage: 12

IMPORT DATA TỪ ANTUTU

In [9]:
antutu = pd.read_csv(r'antutu_socket.csv')
att = antutu.copy()

LẤY THÊM DATA TỪ GSM

In [10]:
gsm = pd.read_csv(r'all_phones_final.csv')
mem = gsm[['name_clean', 'Memory | Internal']]
ref_rate = gsm[['name_clean', 'Display | Type']]
battery = gsm[['name_clean', 'Battery | Type']]

### CLEANING PROCESS

CLEAN NAME

In [11]:
df["Name"] = df["Name"].apply(cln.clean_phone_name)
mem['name_clean'] = mem['name_clean'].apply(cln.clean_phone_name)
ref_rate['name_clean'] = ref_rate['name_clean'].apply(cln.clean_phone_name)
battery['name_clean'] = battery['name_clean'].apply(cln.clean_phone_name)

CLEAN CHIPSET AND ADD CHIPSET INFO TỪ ANTUTU

In [12]:
df['Chipset'] = df['Chipset'].apply(cln.clean_chipset)
df = cln.add_chipset_info(att, df)

In [13]:
specified_features = ['antutu_11', 'clock', 'gpu', 'architecture'] 

non_null_ratios = df[specified_features].notna().mean()

coverage_df = pd.DataFrame({
    'Non-Null Count': df[specified_features].notna().sum(),
    'Coverage Ratio (%)': (non_null_ratios * 100).round(2)
})

display(coverage_df)

,Non-Null Count,Coverage Ratio (%)
antutu_11,749,77.54
clock,749,77.54
gpu,749,77.54
architecture,749,77.54


CLEAN NFC

In [14]:
df['NFC'] = df['NFC'].map(lambda x : 1 if x == "Có" else 0)

### EXTRACTING PROCESS

EXTRACT BRAND

In [15]:
df['Brand'] = df['Name'].apply(cln.get_brand)
df[['Name','Brand']].head()

,Name,Brand
0,iphone 17 pro,apple
1,oppo find x9s,oppo
2,samsung galaxy s26 ultra,samsung
3,iphone 17 pro max,apple
4,samsung galaxy s26,samsung


CLEAN OPERATING SYSTEM

In [16]:
df[["OS_Name", "OS_Version"]] = df.apply(
    lambda row: pd.Series(
        cln.advanced_clean_os(
            row["Operating System"],
            row["Brand"]
        )
    ),
    axis=1
)

EXTRACT METRICS

In [17]:
cols_to_clean = ["Screen Size", "clock"]
for col in cols_to_clean:
    df[col] = df[col].apply(cln.clean_metrics)

In [18]:
df[["Screen Size", "clock"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Screen Size  873 non-null    float64
 1   clock        749 non-null    float64
dtypes: float64(2)
memory usage: 15.2 KB


EXTRACT REFRESH RATE

In [19]:
df['Refresh Rate'] = df.apply(
    lambda row: cln.extract_refresh_rate(row['Tần số quét'], row['Display Features']),
    axis=1
)

In [20]:
ref_rate["Refresh Rate"] = ref_rate["Display | Type"].apply(cln.extract_hz_gsm)
ref_rate['name_clean'] = ref_rate['name_clean'].apply(cln.remove_first_word_if_iphone)
ref_rate.drop(columns= 'Display | Type', inplace=True)

In [21]:
ref_rate_clean = ref_rate[['name_clean', 'Refresh Rate']].drop_duplicates(subset=['name_clean'])

df_merged = pd.merge(
    df,
    ref_rate_clean[["name_clean", "Refresh Rate"]],  
    left_on="Name",
    right_on="name_clean",
    how="left",
    suffixes=("", "_from_ref"),  
)

df_merged["Refresh Rate"] = df_merged["Refresh Rate"].fillna(
    df_merged["Refresh Rate_from_ref"]
)

df_merged.drop(columns=["name_clean", "Refresh Rate_from_ref"], inplace=True)

df = df_merged

ADD RAM

In [22]:
df = cln.add_ram(mem, df)

In [23]:
df['RAM'].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: RAM
Non-Null Count  Dtype
--------------  -----
896 non-null    str  
dtypes: str(1)
memory usage: 7.7 KB


EXTRACT STORAGE

In [24]:
df["RAM"] = df["RAM"].apply(cln.clean_storage)
df["ROM"] = df["ROM"].apply(cln.clean_storage)

In [25]:
df[["RAM", "ROM"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RAM     896 non-null    float64
 1   ROM     915 non-null    float64
dtypes: float64(2)
memory usage: 15.2 KB


EXTRACT BATTERY

In [26]:
df['Battery'] = df['Battery'].apply(cln.clean_battery)
df = cln.fill_iphone_battery(df)

In [27]:
battery["Battery"] = battery["Battery | Type"].apply(cln.extract_battery_gsm)
battery.drop(columns= 'Battery | Type', inplace=True)

In [28]:
battery_clean = battery[['name_clean', 'Battery']].drop_duplicates(subset=['name_clean'])

df_merged = pd.merge(
    df,
    battery_clean[["name_clean", "Battery"]],  
    left_on="Name",
    right_on="name_clean",
    how="left",
    suffixes=("", "_from_ref"),  
)

df_merged["Battery"] = df_merged["Battery"].fillna(
    df_merged["Battery_from_ref"]
)

df_merged.drop(columns=["name_clean", "Battery_from_ref"], inplace=True)

df = df_merged

EXTRACT CPU


In [29]:
df[["total_cores", "max_freq_ghz", "min_freq_ghz", "weighted_mean"]] = df["architecture"].apply(
    lambda x: pd.Series(cln.extract_architecture(x))
)

EXTRACT RESOLUTION

In [30]:
df["Reso_Width"], df["Reso_Height"] = zip(
    *df["Screen Resolution"].apply(cln.extract_res_row)
)

EXTRACT SIM


In [31]:
(
    df["Nano_SIM_Count"],
    df["eSIM_Count"],
    df["Micro_SIM_Count"],
    df["Mini_SIM_Count"],
) = zip(*df["SIM Card"].apply(cln.clean_sim_options))

EXTRACT CAMERA

In [32]:
df = cln.extract_camera_info(df)

EXTRACT DISPLAY

In [33]:

df['Display'] = df['Display'].apply(cln.extract_display_type)

EXTRACT CHIPSET

In [34]:
df['Chipset_name'] = df['Chipset'].apply(cln.extract_chipset)

In [35]:
df['Chipset_gen'] = df['Chipset'].apply(cln.extract_chipset_gen)

EXTRACT GPU

In [36]:
df['gpu_name'] = df['gpu'].apply(cln.extract_gpu)

In [37]:
df['gpu_gen'] = df['gpu'].apply(cln.extract_gpu_gen)

### LOẠI BỎ NHỮNG DÒNG KHÔNG PHẢI LÀ ĐIỆN THOẠI

In [38]:
mask_not_smartphone = (
    (df['Screen Size'] < 4.0) |
    (df['Name'].str.lower().str.contains('tab|pad|win rt', na=False))
)

df = df[~mask_not_smartphone].reset_index(drop=True)
print(f'Đã drop {mask_not_smartphone.sum()} sản phẩm, còn lại {len(df)}')

Đã drop 42 sản phẩm, còn lại 924


### DERIVING PROCESS

Derive PPI

In [39]:
median_screen = df.loc[df['Screen Size'] > 0, 'Screen Size'].median()
df['Screen Size'] = df['Screen Size'].replace(0, median_screen)

df['PPI'] = (
    np.sqrt(df['Reso_Width']**2 + df['Reso_Height']**2) / df['Screen Size']
).round(1)

df.drop(columns=['Reso_Width', 'Reso_Height'], inplace=True)


Derive SIM_total

In [40]:
df['SIM_total'] = (
    df['Nano_SIM_Count'] +
    df['eSIM_Count'] +
    df['Micro_SIM_Count'] +
    df['Mini_SIM_Count']
).clip(upper=2)

df.drop(columns=[ 'Nano_SIM_Count', 'Micro_SIM_Count', 'Mini_SIM_Count'], inplace=True)

Derive eSIM, dien thoai nao co eSIM: 1, khong co: 0

In [41]:
df['has_eSIM'] = (df['eSIM_Count'] > 0).astype(int)

df.drop(columns=['eSIM_Count'], inplace=True)

In [42]:
drop = [
    'Rear Camera', 'Front Camera', 'Screen Resolution', 'Display Features',
    'Operating System', 'SIM Card', 'OS_Is_Android', 'Tần số quét',
    'CPU', 'architecture', 'camera_score', 'Link']

df.drop(columns=drop, inplace = True, errors='ignore')

### TÍNH TỈ LỆ NULL CÒN LẠI

In [43]:
# Tính tỷ lệ % null của từng cột
null_percentages_before = (df.isnull().sum() / df.shape[0]) * 100

# Chỉ lọc ra những cột có % null > 0
null_cols_percentage_before = null_percentages_before[null_percentages_before > 0].sort_values(ascending=False)

print("Tỷ lệ % null theo từng cột:")
print(null_cols_percentage_before.round(2).astype(str) + '%')

Tỷ lệ % null theo từng cột:
front_f/          48.38%
rear_f/           40.91%
PPI               31.49%
OS_Version        27.92%
clock             21.43%
antutu_11         21.43%
max_freq_ghz      21.43%
min_freq_ghz      21.43%
weighted_mean     21.43%
Refresh Rate      21.43%
total_cores       21.43%
gpu               21.43%
front_mp          12.12%
Chipset           11.69%
Screen Size        9.96%
rear_mp_max        9.63%
rear_count         9.52%
rear_telephoto     9.42%
rear_ois           9.42%
rear_wide          9.42%
Battery            6.17%
RAM                 4.0%
ROM                 3.9%
dtype: str


### LỌC RA NHỮNG ĐIỆN THOẠI CÓ ĐỦ DATA VỀ ANTUTU SCORE


In [44]:
df = df[
    df['antutu_11'].notna()  
].copy()

print(len(df))  

726


In [45]:
# Tính tỷ lệ % null của từng cột
null_percentages_after = (df.isnull().sum() / df.shape[0]) * 100

# Chỉ lọc ra những cột có % null > 0
null_cols_percentage_after = null_percentages_after[null_percentages_after > 0].sort_values(ascending=False)

print("Tỷ lệ % null theo từng cột:")
print(null_cols_percentage_after.round(2).astype(str) + '%')

Tỷ lệ % null theo từng cột:
front_f/          48.35%
rear_f/           41.74%
PPI               35.95%
OS_Version        29.75%
Refresh Rate       20.8%
front_mp          15.15%
Chipset           14.88%
Screen Size       12.53%
rear_mp_max       12.12%
rear_count        11.98%
rear_telephoto    11.85%
rear_ois          11.85%
rear_wide         11.85%
Battery            7.85%
ROM                4.82%
RAM                4.68%
dtype: str


### IMPUTATION 

Imputate OS version

In [46]:
df = cln.fill_missing_os_by_brand(
    df, brand_col="Brand", os_col="OS_Version"
)

Imputate Camera by chipset

In [47]:
df = cln.fill_camera_by_chipset(df)

Imputate PPI by Screen Size

In [48]:
df = cln.fill_ppi(df)

Imputate Refresh Rate by Chipset

In [49]:
df = cln.fill_refresh_rate(df)

In [50]:
# Define the core columns (deduplicated to avoid double-counting)
core_cols = ['Screen Size', 'Battery', 'RAM', 'ROM']

# Compute the null count per row
df['core_null_count'] = df[core_cols].isna().sum(axis=1)

Tier 2: Rows with more than 3 null coumns

In [51]:
tier_2 = df.query('core_null_count > 3')
print(tier_2)

                      Name  Screen Size Display Chipset  NFC  ROM  RAM  \
474        nubia neo 5 max          NaN   Other     NaN    0  NaN  NaN   
568       tecno pova 7 neo          NaN   Other     NaN    0  NaN  NaN   
716        tecno camon 40s          NaN   Other     NaN    0  NaN  NaN   
770  samsung galaxy z flip          NaN   Other     NaN    0  NaN  NaN   
801       oneplus 10 ultra          NaN   Other     NaN    0  NaN  NaN   
868             oneplus 10          NaN   Other     NaN    1  NaN  NaN   
872            oneplus 10r          NaN   Other     NaN    1  NaN  NaN   
891  samsung galaxy z fold          NaN   Other     NaN    0  NaN  NaN   
916             oneplus 10          NaN   Other     NaN    1  NaN  NaN   

     Battery  antutu_11   clock  ... front_mp front_f/ Chipset_name  \
474      NaN   419670.0  1850.0  ...      8.0      2.0        Other   
568      NaN   419670.0  1850.0  ...      8.0      2.0        Other   
716      NaN   419670.0  1850.0  ...      8.0 

In [52]:
df_filled = cln.fill_ram_by_chipset(df)
df_filled = cln.fill_rom_by_chipset(df_filled)
df_filled = cln.fill_battery_by_brand(df_filled)
df_filled = cln.fill_screen_size_by_brand_chipset(df_filled)

tier_2 = df_filled.loc[tier_2.index]

In [53]:
tier_2.shape[0]

9

Tier_1: rows with less than 4 null columns

In [54]:
tier_1 = df.query('core_null_count <= 3')

In [55]:
tier_1.shape[0]

717

Imputate Display & Battery features using KNN

In [56]:
display_battery_cols = ['Screen Size', 'PPI', 'Refresh Rate', 'Battery']
a = cln.impute_numeric_with_knn(df, numeric_cols=display_battery_cols, n_neighbors=5)
tier_1 = a.loc[tier_1.index]

Imputate Hardware features using KNN

In [57]:
hardware_cols = ['RAM', 'ROM', 'antutu_11']
a = cln.impute_numeric_with_knn(df, numeric_cols=hardware_cols, n_neighbors=5)
tier_1 = a.loc[tier_1.index]

Impuatate Camera features using KNN

In [58]:
camera_cols = ['rear_count', 'rear_mp_max', 'front_mp', 'rear_ois', 'rear_telephoto', 'rear_wide', 'front_f/', 'rear_f/']
a = cln.impute_numeric_with_knn(df, numeric_cols=camera_cols, n_neighbors=5)
tier_1 = a.loc[tier_1.index]

In [59]:
tier_1.info()

<class 'pandas.DataFrame'>
Index: 717 entries, 0 to 923
Data columns (total 35 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             717 non-null    str    
 1   Screen Size      635 non-null    float64
 2   Display          717 non-null    str    
 3   Chipset          618 non-null    str    
 4   NFC              717 non-null    int64  
 5   ROM              691 non-null    float64
 6   RAM              692 non-null    float64
 7   Battery          669 non-null    float64
 8   antutu_11        717 non-null    float64
 9   clock            717 non-null    float64
 10  gpu              717 non-null    str    
 11  Brand            717 non-null    str    
 12  OS_Name          717 non-null    str    
 13  OS_Version       717 non-null    float64
 14  Refresh Rate     717 non-null    float64
 15  total_cores      717 non-null    float64
 16  max_freq_ghz     717 non-null    float64
 17  min_freq_ghz     717 non-null   

Combine tier_1 and tier_2

In [60]:
df = pd.concat([tier_1, tier_2], axis=0)
df = df.reset_index(drop=True)

In [61]:
df = cln.fill_battery_by_brand(df)
df = cln.fill_screen_size_by_brand_chipset(df)
df = cln.fill_rom_by_chipset(df)
df = cln.fill_ram_by_chipset(df)

In [62]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 726 entries, 0 to 725
Data columns (total 35 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             726 non-null    str    
 1   Screen Size      726 non-null    float64
 2   Display          726 non-null    str    
 3   Chipset          618 non-null    str    
 4   NFC              726 non-null    int64  
 5   ROM              726 non-null    float64
 6   RAM              726 non-null    float64
 7   Battery          726 non-null    float64
 8   antutu_11        726 non-null    float64
 9   clock            726 non-null    float64
 10  gpu              726 non-null    str    
 11  Brand            726 non-null    str    
 12  OS_Name          726 non-null    str    
 13  OS_Version       726 non-null    float64
 14  Refresh Rate     726 non-null    float64
 15  total_cores      726 non-null    float64
 16  max_freq_ghz     726 non-null    float64
 17  min_freq_ghz     726 non-nu

In [63]:
df[df['Refresh Rate'] > 144.0][['Name', 'Refresh Rate']]


,Name,Refresh Rate
159,asus rog phone 6,165.0
290,xiaomi 17 pro max,2160.0
300,xiaomi 17,2160.0
313,sharp aquos r8,240.0
315,xiaomi 17 pro,2160.0
321,leitz phone 1,240.0
350,motorola razr 40 ultra,165.0
475,motorola edge 40 pro,165.0
636,asus rog phone 8,165.0
646,rog phone 6 pro,165.0


In [64]:
df.loc[df['Name'].str.contains('xiaomi 17', case=False, na=False), 'Refresh Rate'] = 120.0

In [65]:
drop_cols = ['gpu', 'Chipset', 'core_null_count']

df.drop(columns = drop_cols, inplace=True)

In [66]:
mask_not_smartphone = (          
    (df['RAM'] < 1.0) |                
    (df['ROM'] < 1.0)                
)

print(f'Số sản phẩm không phải smartphone: {mask_not_smartphone.sum()}')
print(df.loc[mask_not_smartphone, ['Name', 'Screen Size', 'RAM', 'ROM']].to_string())

Số sản phẩm không phải smartphone: 1
                Name  Screen Size  RAM   ROM
674  google pixel 8a          6.1  8.0  0.25


In [67]:
df.loc[df['Name'].str.strip().str.lower() == 'google pixel 8a', 'ROM'] = 256

In [68]:
df= df.groupby('Name').agg(
    **{col: (col, 'first') for col in df.columns if col not in ['Name', 'RAM', 'ROM']},
    RAM_min=('RAM', 'min'),
    RAM_max=('RAM', 'max'),
    ROM_min=('ROM', 'min'),
    ROM_max=('ROM', 'max'),
).reset_index()

In [69]:
df.duplicated('Name').sum()

np.int64(0)

In [70]:
df.to_csv('preprocess_1.csv')

In [71]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 578 entries, 0 to 577
Data columns (total 34 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            578 non-null    str    
 1   Screen Size     578 non-null    float64
 2   Display         578 non-null    str    
 3   NFC             578 non-null    int64  
 4   Battery         578 non-null    float64
 5   antutu_11       578 non-null    float64
 6   clock           578 non-null    float64
 7   Brand           578 non-null    str    
 8   OS_Name         578 non-null    str    
 9   OS_Version      578 non-null    float64
 10  Refresh Rate    578 non-null    float64
 11  total_cores     578 non-null    float64
 12  max_freq_ghz    578 non-null    float64
 13  min_freq_ghz    578 non-null    float64
 14  weighted_mean   578 non-null    float64
 15  rear_count      578 non-null    float64
 16  rear_mp_max     578 non-null    float64
 17  rear_f/         578 non-null    float64
 18  r